# Amazon Bedrock AgentCore 메모리를 활용한 메모리 레코드 이벤트 스트리밍

## 개요

이 튜토리얼은 Amazon Bedrock AgentCore 메모리로 [**메모리 레코드 스트리밍**](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-record-streaming.html)을 설정하는 방법을 보여줍니다. [Amazon Kinesis Data Stream](https://docs.aws.amazon.com/streams/latest/dev/introduction.html)을 구성하여 메모리 레코드가 생성, 업데이트 또는 삭제될 때 실시간 알림을 수신합니다 - API 폴링 없이 이벤트 기반 아키텍처를 가능하게 합니다.

### 튜토리얼 세부 정보

| 항목                | 세부 내용                                                        |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | 메모리 레코드 스트리밍                                           |
| 기능                | 장기 메모리 이벤트 스트리밍                                      |
| 주요 기능           | Kinesis Data Streams, 메모리 레코드 라이프사이클 이벤트          |
| 난이도              | 중급                                                             |
| 사용 SDK            | boto3                                                            |

### 학습 내용

이 튜토리얼에서 배울 내용:
1. 메모리 레코드 이벤트를 수신하기 위한 Amazon Kinesis Data Stream 생성
2. AgentCore 메모리가 스트림에 게시할 수 있도록 IAM 역할 설정
3. 스트리밍이 활성화된 메모리 리소스 생성
4. 실시간으로 메모리 레코드 라이프사이클 이벤트 트리거 및 소비
5. 이벤트 콘텐츠 수준(`FULL_CONTENT` 또는 `METADATA_ONLY`) 구성
   
### 작동 방식

메모리 레코드 스트리밍은 푸시 기반 전달 모델을 사용합니다. 메모리 레코드가 변경되면 이벤트가 자동으로 Kinesis Data Stream에 게시됩니다:

- **MemoryRecordCreated** - 장기 메모리 추출 또는 `BatchCreateMemoryRecords` API에 의해 트리거
- **MemoryRecordUpdated** - `BatchUpdateMemoryRecords` API에 의해 트리거
- **MemoryRecordDeleted** - 통합 워크플로우, `DeleteMemoryRecord` 또는 `BatchDeleteMemoryRecords` API에 의해 트리거

### 아키텍처

<div style="text-align:left">
    <img src="memory_record_streaming.png" width="90%"/>
</div>

## 0. 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
* Python 3.10+
* AgentCore 메모리, Kinesis 및 IAM에 접근할 수 있도록 구성된 AWS 자격 증명
* Amazon Bedrock 모델 접근 (장기 메모리 추출용)

먼저 필요한 라이브러리를 설치하겠습니다:

In [ ]:
!pip install boto3>=1.42.63

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성하겠습니다:

In [ ]:
import os
import json
import time
import uuid
import base64
import logging
import boto3
from datetime import datetime, timezone
from botocore.exceptions import ClientError

# 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("memory-streaming")

REGION = os.getenv('AWS_REGION', 'us-west-2')
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

# boto3 클라이언트 초기화
kinesis_client = boto3.client('kinesis', region_name=REGION)
iam_client = boto3.client('iam')
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)

# 리소스 이름 지정을 위한 고유 식별자
unique_id = str(uuid.uuid4())[:8]
print(f"계정: {ACCOUNT_ID}, 리전: {REGION}, 고유 ID: {unique_id}")

## 1. Kinesis Data Stream 생성

먼저 계정에 Kinesis Data Stream을 생성합니다. AgentCore 메모리가 메모리 레코드 라이프사이클 이벤트를 게시할 위치입니다.

이 튜토리얼에서는 단일 샤드(초당 최대 1000 레코드 쓰기 지원)를 사용하며 충분합니다. 프로덕션 워크로드의 경우 용량 확장을 위해 [스트림 리샤딩](https://docs.aws.amazon.com/streams/latest/dev/kinesis-using-sdk-java-resharding.html)을 확인하세요.

In [ ]:
stream_name = f"memory-record-stream-{unique_id}"

try:
    kinesis_client.create_stream(
        StreamName=stream_name,
        ShardCount=1  # 이 튜토리얼에는 단일 샤드로 충분
    )
    logger.info(f"Kinesis 스트림 생성 중: {stream_name}")

    # 스트림이 활성화될 때까지 대기
    waiter = kinesis_client.get_waiter('stream_exists')
    waiter.wait(StreamName=stream_name)

    stream_info = kinesis_client.describe_stream(StreamName=stream_name)
    stream_arn = stream_info['StreamDescription']['StreamARN']
    print(f"Kinesis 스트림 생성 완료: {stream_arn}")

except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceInUseException':
        stream_info = kinesis_client.describe_stream(StreamName=stream_name)
        stream_arn = stream_info['StreamDescription']['StreamARN']
        print(f"스트림이 이미 존재합니다: {stream_arn}")
    else:
        raise

## 2. 스트리밍을 위한 IAM 역할 생성

AgentCore 메모리는 Kinesis Data Stream에 이벤트를 게시하기 위해 수임할 수 있는 IAM 역할이 필요합니다. 이 역할에는 다음이 필요합니다:
- `bedrock-agentcore.amazonaws.com`이 역할을 수임할 수 있도록 하는 **신뢰 정책**
- 스트림에 대해 `kinesis:PutRecords` 및 `kinesis:DescribeStream`을 허용하는 **권한 정책**

In [ ]:
role_name = f"AgentCoreMemoryStreamingRole-{unique_id}"

# 신뢰 정책 - AgentCore 메모리가 이 역할을 수임할 수 있도록 허용
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

# 권한 정책 - 특정 Kinesis 스트림으로 범위 제한
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "kinesis:PutRecords",
                "kinesis:DescribeStream"
            ],
            "Resource": stream_arn
        }
    ]
}

try:
    # IAM 역할 생성
    create_role_response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Allows AgentCore Memory to publish events to Kinesis"
    )
    role_arn = create_role_response['Role']['Arn']

    # 인라인 권한 정책 연결
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName="KinesisPublishPolicy",
        PolicyDocument=json.dumps(permissions_policy)
    )

    # IAM 전파를 위한 대기 시간
    print(f"IAM 역할 생성 완료: {role_arn}")
    print("IAM 전파를 위해 10초 대기 중...")
    time.sleep(10)

except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{role_name}"
        print(f"역할이 이미 존재합니다: {role_arn}")
    else:
        raise

## 3. 스트리밍이 활성화된 메모리 생성

이제 스트림 전달 구성이 포함된 AgentCore 메모리 리소스를 생성합니다. 핵심 파라미터는:

- **`streamDeliveryResources`** - Kinesis 스트림을 가리키고 콘텐츠 수준을 지정
- **`memoryExecutionRoleArn`** - AgentCore가 이벤트 게시를 위해 수임할 IAM 역할
- **`FULL_CONTENT`** - 각 이벤트에 메모리 레코드 텍스트를 포함 (경량 알림만 필요한 경우 `METADATA_ONLY` 사용)

또한 대화 이벤트가 메모리 레코드로 추출되고, 이어서 스트리밍 이벤트를 트리거하도록 장기 메모리 전략(사용자 선호도)을 구성합니다.

In [ ]:
memory_name = "streaming_memory"
actor_id = "demo-user"

def wait_for_memory_active(memory_id, timeout=250):
    """메모리 상태가 ACTIVE가 되거나 타임아웃까지 GetMemory를 폴링합니다."""
    start = time.time()
    while time.time() - start < timeout:
        resp = agentcore_control_client.get_memory(memoryId=memory_id)
        status = resp['memory']['status']
        print(f"  메모리 상태: {status}")
        if status == 'ACTIVE':
            return resp['memory']
        if status == 'FAILED':
            raise RuntimeError(f"메모리 생성 실패: {resp['memory'].get('failureReason')}")
        time.sleep(5)
    raise TimeoutError("메모리가 ACTIVE 상태가 되기를 기다리는 중 타임아웃")

try:
    response = agentcore_control_client.create_memory(
        name=memory_name,
        description="Memory with long-term memory record streaming enabled",
        eventExpiryDuration=7,
        memoryExecutionRoleArn=role_arn,
        memoryStrategies=[
            {
                "userPreferenceMemoryStrategy": {
                    "name": "UserPreferences",
                    "description": "Extracts user preferences, facts, and interests from conversations",
                    "namespaces": ["/{actorId}/user_preferences/"],
                }
            }
        ],
        streamDeliveryResources={
            "resources": [
                {
                    "kinesis": {
                        "dataStreamArn": stream_arn,
                        "contentConfigurations": [
                            {
                                "type": "MEMORY_RECORDS",
                                "level": "FULL_CONTENT"
                            }
                        ]
                    }
                }
            ]
        }
    )
    memory_id = response['memory']['id']
    print(f"메모리 생성 시작됨: {memory_id}")
    print("메모리가 ACTIVE 상태가 되기를 기다리는 중...")
    wait_for_memory_active(memory_id)
    print(f"스트리밍이 활성화된 메모리 생성 완료: {memory_id}")

except ClientError as e:
    logger.error(f"메모리 생성 오류: {e}")
    raise

## 4. 스트리밍 활성화 확인

스트리밍이 포함된 메모리를 생성하면 AgentCore 메모리가 구성을 검증하고 Kinesis 스트림에 `StreamingEnabled` 이벤트를 게시합니다. 스트림에서 읽어 확인해 보겠습니다.

In [ ]:
def read_kinesis_events(stream_name, max_wait_seconds=60, max_events=10):
    """Kinesis Data Stream에서 이벤트를 읽습니다.
    
    스트림에서 새 레코드를 폴링하고 디코딩합니다.
    
    Args:
        stream_name: Kinesis 스트림 이름
        max_wait_seconds: 반환하기 전 최대 폴링 시간
        max_events: 수집할 최대 이벤트 수
    
    Returns:
        디코딩된 이벤트 페이로드 목록
    """
    events = []
    
    # 가장 오래된 사용 가능한 레코드부터 시작하는 샤드 반복자 가져오기
    stream_info = kinesis_client.describe_stream(StreamName=stream_name)
    shard_id = stream_info['StreamDescription']['Shards'][0]['ShardId']
    
    iterator_response = kinesis_client.get_shard_iterator(
        StreamName=stream_name,
        ShardId=shard_id,
        ShardIteratorType='TRIM_HORIZON' # 가장 오래된 사용 가능한 레코드부터 읽기
    )
    shard_iterator = iterator_response['ShardIterator']
    
    start_time = time.time()
    while time.time() - start_time < max_wait_seconds and len(events) < max_events:
        response = kinesis_client.get_records(
            ShardIterator=shard_iterator,
            Limit=100
        )
        
        for record in response['Records']:
            data = base64.b64decode(record['Data']) if isinstance(record['Data'], str) else record['Data']
            payload = json.loads(data)
            events.append(payload)
        
        shard_iterator = response['NextShardIterator']
        
        if not response['Records']:
            time.sleep(2)
    
    return events

In [ ]:
# StreamingEnabled 검증 이벤트 확인
print("StreamingEnabled 이벤트 확인 중...")
events = read_kinesis_events(stream_name, max_wait_seconds=30, max_events=1)

if events:
    for event in events:
        print(json.dumps(event, indent=2))
else:
    print("아직 이벤트가 수신되지 않았습니다. StreamingEnabled 이벤트가 도착하는 데 시간이 걸릴 수 있습니다.")

## 5. 메모리 레코드 이벤트 트리거

이제 대화 데이터를 생성하여 메모리 레코드 라이프사이클 이벤트를 발생시켜 보겠습니다. 두 가지 접근 방식을 사용합니다:

| 접근 방식 | API | 작동 방식 |
|:---------|:----|:-------------|
| **옵션 A** | [`CreateEvent`](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_CreateEvent.html) | 대화를 전송하면 AgentCore가 구성된 전략을 통해 **비동기적으로** 장기 레코드를 추출 |
| **옵션 B** | [`BatchCreateMemoryRecords`](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_BatchCreateMemoryRecords.html) | 레코드를 직접 생성하면 이벤트가 **즉시** 게시 |

### 옵션 A: 단기 메모리를 통한 이벤트 생성 (비동기 추출 트리거)

대화 이벤트를 전송하면 AgentCore 메모리가 구성된 전략을 사용하여 비동기적으로 장기 메모리 레코드를 추출합니다. 추출된 각 레코드는 스트림에 `MemoryRecordCreated` 이벤트를 트리거합니다.

In [ ]:
# 추출 가능한 사용자 선호도가 포함된 대화 전송
agentcore_client.create_event(
    memoryId=memory_id,
    actorId=f"{actor_id}",
    sessionId="streaming-demo-session-001",
    eventTimestamp=datetime.now(timezone.utc),
    payload=[
        {
            "conversational": {
                "content": {"text": "I went hiking yesterday. I really like hiking in the Pacific Northwest. I also enjoyed the Thai restaurant. Thai food is amazing."},
                "role": "USER"
            }
        },
        {
            "conversational": {
                "content": {"text": "Great! I'll remember that you enjoy hiking in the Pacific Northwest and prefer Thai dining options."},
                "role": "ASSISTANT"
            }
        }
    ]
)

print("대화 이벤트가 전송되었습니다. 장기 메모리 추출이 비동기적으로 수행됩니다.")

### 옵션 B: 메모리 레코드 직접 생성

`BatchCreateMemoryRecords` API를 사용하여 메모리 레코드를 직접 생성할 수도 있습니다. 생성된 각 레코드는 즉시 `MemoryRecordCreated` 이벤트를 트리거합니다.

In [ ]:
response = agentcore_client.batch_create_memory_records(
    memoryId=memory_id,
    records=[
        {
            "requestIdentifier": "direct-record-1",
            "content": {"text": "User prefers window seats on flights"},
            "namespaces": [f"/{actor_id}/user_preferences/"],
            "timestamp": str(int(time.time()))
        },
        {
            "requestIdentifier": "direct-record-2",
            "content": {"text": "User's favorite programming language is Python"},
            "namespaces": [f"/{actor_id}/user_preferences/"],
            "timestamp": str(int(time.time()))
        }
    ]
)

print(f"배치로 {len(response.get('successfulRecords', []))}개의 메모리 레코드를 직접 생성했습니다.")
print(f"레코드 ID: \n {[i.get('memoryRecordId') for i in response.get('successfulRecords', [])]}")

## 6. 스트리밍 이벤트 소비

Kinesis 스트림에서 읽어 메모리 레코드 라이프사이클 이벤트를 확인하겠습니다. 장기 메모리 추출이 비동기적이므로, 처리 시간을 위해 최대 90초 동안 폴링합니다.


> **프로덕션 참고사항:** 실제 애플리케이션에서는 폴링 대신 [AWS Lambda 이벤트 소스 매핑](https://docs.aws.amazon.com/lambda/latest/dg/with-kinesis.html) 또는 [Amazon Kinesis Client Library(KCL)](https://docs.aws.amazon.com/streams/latest/dev/shared-throughput-kcl-consumers.html)을 사용하여 스트림을 소비할 수 있습니다.

In [ ]:
print("Kinesis 스트림에서 메모리 레코드 이벤트 폴링 중...\n")
events = read_kinesis_events(stream_name, max_wait_seconds=90, max_events=10)

print(f"{len(events)}개의 이벤트 수신:\n")
for i, event in enumerate(events):
    stream_event = event.get("memoryStreamEvent", {})
    event_type = stream_event.get("eventType", "Unknown")
    event_time = stream_event.get("eventTime", "")
    record_id = stream_event.get("memoryRecordId", "N/A")
    record_text = stream_event.get("memoryRecordText", "")
    
    print(f"--- 이벤트 {i+1}: {event_type} ---")
    print(f"  시간:      {event_time}")
    print(f"  메모리 ID: {stream_event.get('memoryId', 'N/A')}")
    print(f"  레코드 ID: {record_id}")
    if record_text:
        print(f"  콘텐츠:    {record_text[:120]}...")
    print()

### 전체 이벤트 페이로드 검사

한 이벤트의 원시 JSON을 확인하여 전체 스키마를 살펴보겠습니다:

In [ ]:
if events:
    # 첫 번째 MemoryRecordCreated 이벤트의 전체 페이로드 표시
    created_events = [e for e in events if e.get("memoryStreamEvent", {}).get("eventType") == "MemoryRecordCreated"]
    if created_events:
        print("MemoryRecordCreated 이벤트 전체 페이로드:")
        print(json.dumps(created_events[0], indent=2))
    else:
        print("첫 번째 이벤트의 전체 페이로드:")
        print(json.dumps(events[0], indent=2))
else:
    print("검사할 이벤트가 없습니다. 추출이 아직 진행 중일 수 있습니다 - 이 셀을 다시 실행해 보세요.")

## 7. ListMemoryRecords와 교차 검증

스트리밍된 이벤트가 메모리에 저장된 것과 일치하는지 레코드를 직접 나열하여 확인하겠습니다:

In [ ]:
records_response = agentcore_client.list_memory_records(
    memoryId=memory_id,
    namespace=f"/{actor_id}/user_preferences/"
)

records = records_response.get('memoryRecordSummaries', [])
print(f"네임스페이스 '/{actor_id}/user-preferences/'에서 {len(records)}개의 메모리 레코드 발견:\n")

for record in records:
    print(f"  레코드 ID: {record['memoryRecordId']}")
    print(f"  콘텐츠:    {record.get('content', {}).get('text', 'N/A')[:120]}")
    print(f"  생성일:    {record.get('createdAt', 'N/A')}")
    print()

## 8. 정리 (선택 사항)

실험이 끝나면 이 튜토리얼에서 생성한 리소스를 정리하세요:

> **비용 참고사항:** Kinesis Data Streams는 [샤드당 시간당 요금](https://aws.amazon.com/kinesis/data-streams/pricing/)이 발생합니다. 지속적인 비용을 방지하려면 완료 후 스트림을 삭제하세요.

In [ ]:
# 메모리 리소스 삭제
try:
    agentcore_control_client.delete_memory(memoryId=memory_id)
    print(f"메모리 삭제 중: {memory_id}")
    # 삭제가 완료될 때까지 폴링
    while True:
        try:
            resp = agentcore_control_client.get_memory(memoryId=memory_id)
            status = resp['memory']['status']
            print(f"  메모리 상태: {status}")
            if status == 'DELETING':
                time.sleep(5)
            else:
                break
        except agentcore_control_client.exceptions.ResourceNotFoundException:
            print(f"  메모리가 성공적으로 삭제되었습니다.")
            break
except Exception as e:
    print(f"메모리 삭제 오류: {e}")

# Kinesis 스트림 삭제
try:
    kinesis_client.delete_stream(StreamName=stream_name, EnforceConsumerDeletion=True)
    print(f"Kinesis 스트림 삭제 완료: {stream_name}")
except Exception as e:
    print(f"스트림 삭제 오류: {e}")

# IAM 역할 삭제 (먼저 인라인 정책을 제거해야 함)
try:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName="KinesisPublishPolicy")
    iam_client.delete_role(RoleName=role_name)
    print(f"IAM 역할 삭제 완료: {role_name}")
except Exception as e:
    print(f"IAM 역할 삭제 오류: {e}")

## 결론

이 튜토리얼에서 Amazon Bedrock AgentCore 메모리를 사용한 엔드투엔드 메모리 레코드 스트리밍을 설정했습니다. 배운 내용:

1. **Kinesis Data Stream 생성** - 메모리 레코드 라이프사이클 이벤트를 수신
2. **IAM 역할 구성** - AgentCore가 스트림에 게시할 수 있도록 최소 권한 부여
3. **메모리 리소스 생성** - 스트리밍 활성화 및 `FULL_CONTENT` 전달 설정
4. **이벤트 트리거** - 대화 추출 및 직접 레코드 생성 모두 활용
5. **이벤트 소비 및 검사** - 스트림에서 실시간으로 `MemoryRecordCreated` 이벤트 확인

### 다음 단계
AgentCore 메모리 스트리밍 기능을 더 잘 활용하려면 다음을 시도해 볼 수 있습니다:
- **Lambda 소비자 추가** - 이벤트를 자동으로 처리 (예제는 [문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/userguide/memory-streaming.html) 참조)
- **`METADATA_ONLY`로 전환** - 변경 알림만 필요할 때 데이터 전송 줄이기
- **CloudWatch 알람 설정** - 프로덕션 모니터링을 위해 `StreamPublishingFailure` 및 `StreamUserError` 메트릭 사용
- **이벤트 기반 워크플로우 구축** - 메모리 레코드를 S3의 데이터 레이크에 동기화, 알림 트리거, 또는 다운스트림 사용자 프로필 업데이트